In [ ]:
import polars as pl
import duckdb
from pathlib import Path
import xml.etree.ElementTree as ET
DATA_PATH="~/health/data/raw/export.xml"


強みは、「Polarsでデータを読み込んで整え、それをそのままDuckDBに一発で放り込める」 という点


In [ ]:
file_path = DATA_PATH



In [ ]:
df_inbody = pl.read_csv("~/health/data/raw/inbody.5-10csv")

In [ ]:
import xml.etree.ElementTree as ET
import polars as pl

def parse_apple_health_xml(xml_path: str) -> pl.DataFrame:
    records = []
    
    # 抽出したい項目の識別子マッピング
    target_types = {
        'HKQuantityTypeIdentifierStepCount': 'step_count',                # 歩数
        'HKQuantityTypeIdentifierActiveEnergyBurned': 'active_calories', # 運動消費カロリー
        'HKQuantityTypeIdentifierBasalEnergyBurned': 'resting_calories',  # 基礎代謝消費
        'HKQuantityTypeIdentifierHeartRate': 'heart_rate',                # 心拍数
    }
    
    # 巨大なXMLを1要素ずつストリーミング読み込み
    context = ET.iterparse(xml_path, events=('end',))
    for event, elem in context:
        if elem.tag == 'Record':
            record_type = elem.attrib.get('type')
            if record_type in target_types:
                records.append({
                    'type': target_types[record_type],
                    'value': float(elem.attrib.get('value', 0)),
                    # 日付文字列から先頭10文字（YYYY-MM-DD）を取得
                    'date': elem.attrib.get('startDate', '')[:10]
                })
            # メモリ解放
            elem.clear()
            
    # 辞書のリストから一括で Polars DataFrame を作成
    return pl.DataFrame(records)

# 実行例
# df_apple_raw = parse_apple_health_xml("../data/raw/export.xml")

In [ ]:
import xml.etree.ElementTree as ET
import polars as pl

def parse_all_apple_health_xml(xml_path: str) -> pl.DataFrame:
    records = []
    
    context = ET.iterparse(xml_path, events=('end',))
    
    for event, elem in context:
        if elem.tag == 'Record':
            attrib = elem.attrib
            raw_type = attrib.get('type', '')
            
            # タイプ名から長いプレフィックスを削除
            clean_type = (
                raw_type.replace('HKQuantityTypeIdentifier', '')
                        .replace('HKCategoryTypeIdentifier', '')
                        .replace('HKDataType', '')
            )
            
            # 【調整箇所 1】value は型変換せず文字列のまま取得
            records.append({
                'type': clean_type,
                'value': attrib.get('value'),
                'unit': attrib.get('unit'),
                'date': attrib.get('startDate', '')[:10],  # YYYY-MM-DD
                'source': attrib.get('sourceName')
            })
            
            elem.clear()
            
    # 【調整箇所 2】全列を文字列型（pl.String）として schema を定義
    schema = {
        'type': pl.String,
        'value': pl.String,
        'unit': pl.String,
        'date': pl.String,
        'source': pl.String
    }
    
    # schema を渡して DataFrame を作成
    return pl.DataFrame(records, schema=schema)

In [ ]:
df_all = parse_all_apple_health_xml("data/raw/export.xml")
df_all = df_all.with_columns(
    pl.col("value").cast(pl.Float64, strict=False).alias("value_num")
)

In [ ]:
print(df_all.head())

In [ ]:
type_summary = (
                df_all
                .group_by('type')
                .agg([
                    pl.len().alias("total_count"),
                    pl.col("value_num").is_not_null().sum().alias("numeric_count"),
                    pl.col("unit").filter(pl.col("unit").is_not_null()).first().alias("unit")
                ])
                .sort("total_count",descending=True)
            )
with pl.Config(tbl_rows=100):
    print(type_summary)

In [ ]:
pl.col("numeric_count").is_not_null().sum().alias("neko")